In [37]:
import pandas as pd

In [38]:
df = pd.read_csv(r"C:\Users\hp\Desktop\Amazon\CSV Files\amazon_india_2019.csv")

In [3]:
df.shape

(121605, 34)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121605 entries, 0 to 121604
Data columns (total 34 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   transaction_id          121605 non-null  object 
 1   order_date              121605 non-null  object 
 2   customer_id             121605 non-null  object 
 3   product_id              121605 non-null  object 
 4   product_name            121605 non-null  object 
 5   category                121605 non-null  object 
 6   subcategory             121605 non-null  object 
 7   brand                   121605 non-null  object 
 8   original_price_inr      121605 non-null  object 
 9   discount_percent        121605 non-null  float64
 10  discounted_price_inr    121605 non-null  float64
 11  quantity                121605 non-null  int64  
 12  subtotal_inr            121605 non-null  float64
 13  delivery_charges        111884 non-null  float64
 14  final_amount_inr    

In [39]:
import numpy as np

df.replace("", np.nan, inplace=True)


In [40]:
dfc = df.copy()

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [46]:
import pandas as pd

# Ensure raw strings (very important)
dfc['order_date'] = dfc['order_date'].astype('string').str.strip()

# Explicitly parse DD/MM/YYYY and similar
dfc['order_date'] = pd.to_datetime(
    dfc['order_date'],
    dayfirst=True,
    errors='coerce'
)

# Standardize output format
dfc['order_date'] = dfc['order_date'].dt.strftime('%Y-%m-%d')


In [44]:
dfc['order_date'] = df['order_date']

In [ ]:
dfc['order_date'].head(20)

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees.

In [9]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
        .astype(str)                      
        .str.replace('₹', '', regex=False) 
        .str.replace(',', '', regex=False)
        .str.replace('Rs ', '', regex=False)
        .str.strip()                
)

dfc['original_price_inr'] = pd.to_numeric(
    dfc['original_price_inr']
)


Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.

In [10]:
import re

def parse_rating(r):
    if pd.isna(r):
        return np.nan

    if '/' in r:
        a, b = r.split('/')
        return float(a) / float(b) * 5

    m = re.search(r'\d+\.?\d*', r)
    return float(m.group()) if m else np.nan


dfc['customer_rating'] = dfc['customer_rating'].apply(parse_rating)

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.

In [11]:
dfc['customer_city'] = (
    dfc['customer_city']
    .str.lower()
    .str.strip()
)
 
city_map = {
    'bangalore': 'Bengaluru',
    'bengaluru': 'Bengaluru',
    'bangalore/bengaluru': 'Bengaluru',
    'bengalore' : 'Bengaluru',
    'Bengaluru' : 'banglore',
    
    'mumbai': 'Mumbai',
    'bombay': 'Mumbai',
    'mumbai/bombay': 'Mumbai',
    'mumba' : 'Mumbai',
    'calcutta' : 'kolkata',

    'delhi': 'Delhi',
    'new delhi': 'Delhi',
    'delhi/new delhi': 'Delhi',
    'delhi NCR' : 'Delhi',
    'delhi ncr' : 'Delhi',

    'chenai' : 'chennai',
    'madras' : 'chennai'
}

dfc['customer_city'] = dfc['customer_city'].replace(city_map)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [12]:
import numpy as np

bool_cols = ['is_prime_member', 'is_prime_eligible', 'is_festival_sale']

for col in bool_cols:
    dfc[col] = dfc[col].replace(['', ' ', 'NA', 'N/A', None, 'None'], np.nan)


bool_map = {
    True: True,
    'True': True,
    'true': True,
    'Yes': True,
    'yes': True,
    'Y': True,
    'y': True,
     1: True,
    
     False: False,
    'False': False,
    'false': False,
    'No': False,
    'no': False,
    'N': False,
    'n': False,
     0: False
}


for col in bool_cols:
    dfc[col] = dfc[col].map(bool_map)

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.

In [13]:
dfc.columns = dfc.columns.str.strip()

category_map = {
    'electronics': 'Electronics',
    'ELECTRONICS': 'Electronics',
    'electronics & accessories': 'Electronics',
    'Electronicss': 'Electronics',
    'Electronics & Accessories': 'Electronics',
    'Electronic': 'Electronics',
    'clothing': 'Fashion'
}

dfc['category'] = dfc['category'].replace(category_map)

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [14]:
import numpy as np

days_map = {
    'Express': '0',
    'Same Day': '0',
    '-1': 'None',
    '1-2 days': '2'
}

dfc['delivery_days'] = dfc['delivery_days'].replace(days_map)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [15]:
dup_cols = [
    "customer_id",
    "product_id",
    "order_date",
    "final_amount_inr"
]

price_cols = [
    "original_price_inr",
    "discounted_price_inr",
    "subtotal_inr",
    "final_amount_inr"
]

dfc["dup_count"] = (
    dfc.groupby(dup_cols)["transaction_id"]
      .transform("count")
)


dfc["price_identical"] = (
    dfc.groupby(dup_cols)[price_cols]
      .transform("nunique")
      .max(axis=1) == 1
)


dfc["is_high_value"] = dfc["final_amount_inr"] > 5000
dfc["is_bulk_customer"] = dfc["customer_spending_tier"].isin(["Premium"])
dfc["is_bulk_quantity"] = dfc["quantity"] > 1

dfc["is_duplicate_candidate"] = dfc["dup_count"] > 1


In [16]:
df_deduped = dfc[dfc["is_duplicate_candidate"]].drop_duplicates(subset=dup_cols, keep="first")


In [17]:
print("Rows deleted:", (df_deduped))

Rows deleted:            transaction_id  order_date         customer_id   product_id  \
770     TXN_2019_00000771  2019-12-01  CUST_2019_00041038  PROD_000167   
1544    TXN_2019_00001545  2019-07-01  CUST_2019_00024901  PROD_000069   
1972    TXN_2019_00001973  2019-08-01  CUST_2018_00017978  PROD_001538   
2171    TXN_2019_00002172  2019-04-01  CUST_2018_00016063  PROD_000032   
3191    TXN_2019_00003192  2019-02-01  CUST_2019_00026348  PROD_000461   
...                   ...         ...                 ...          ...   
116597  TXN_2019_00116598  2019-12-12  CUST_2019_00042574  PROD_001745   
117568  TXN_2019_00117569  2019-07-12  CUST_2019_00031535  PROD_000618   
119681  TXN_2019_00119682  2019-10-12  CUST_2019_00029483  PROD_001711   
119728  TXN_2019_00119729  2019-11-12  CUST_2016_00019778  PROD_000317   
119777  TXN_2019_00119778  2019-10-12  CUST_2018_00003241  PROD_000386   

                             product_name     category  subcategory    brand  \
770         OnePl

In [56]:
len(dfc)

77385

In [18]:
cols_to_drop = [
    "dup_count",
    "price_identical",
    "is_high_value",
    "is_bulk_customer",
    "is_bulk_quantity",
    "is_duplicate_candidate"
]

dfc = dfc.drop(columns=cols_to_drop)

Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [19]:
import numpy as np

dfc["product_median_price"] = (
    dfc.groupby("product_id")["final_amount_inr"]
      .transform("median")
)

dfc["price_outlier"] = (
    dfc["final_amount_inr"] > 50 * dfc["product_median_price"]
)

dfc.loc[dfc["price_outlier"], "final_amount_inr"] /= 100
dfc.loc[dfc["price_outlier"], "discounted_price_inr"] /= 100
dfc.loc[dfc["price_outlier"], "original_price_inr"] /= 100

dfc["subtotal_inr"] = dfc["discounted_price_inr"] * dfc["quantity"]
dfc["final_amount_inr"] = dfc["subtotal_inr"] + dfc["delivery_charges"].fillna(0)

dfc["price_corrected_flag"] = dfc["price_outlier"]

dfc.drop(columns=["product_median_price"], inplace=True)

corrected_rows = dfc[dfc["price_corrected_flag"]]

print(corrected_rows)


Empty DataFrame
Columns: [transaction_id, order_date, customer_id, product_id, product_name, category, subcategory, brand, original_price_inr, discount_percent, discounted_price_inr, quantity, subtotal_inr, delivery_charges, final_amount_inr, customer_city, customer_state, customer_tier, customer_spending_tier, customer_age_group, payment_method, delivery_days, delivery_type, is_prime_member, is_festival_sale, festival_name, customer_rating, return_status, order_month, order_year, order_quarter, product_weight_kg, is_prime_eligible, product_rating, price_outlier, price_corrected_flag]
Index: []

[0 rows x 36 columns]


In [20]:
dfc = dfc.drop(
    columns=[
        "price_outlier",
        "price_corrected_flag"       
    ]
)



Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [22]:
dfc["payment_method"] = (
    dfc["payment_method"]
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.strip()
)

payment_map = {
    "UPI": "UPI",
    "PHONEPE": "UPI",
    "GOOGLEPAY": "UPI",
    "GPAY": "UPI",
    "PAYTM": "UPI",

    "CREDIT CARD": "Credit Card",
    "CREDIT_CARD": "Credit Card",
    "CC": "Credit Card",

    "DEBIT CARD": "Debit Card",
    "DC": "Debit Card",

    "COD": "Cash on Delivery",
    "CASH ON DELIVERY": "Cash on Delivery",

    "NET BANKING": "Net Banking"
}

dfc["payment_method"] = dfc["payment_method"].replace(payment_map)

In [23]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
    .astype(str)
    .str.replace('-', '', regex=False)
    .astype(float)
)


In [24]:
dfc["payment_method"].unique()

array(['UPI', 'Cash on Delivery', 'Debit Card', 'Credit Card',
       'Net Banking'], dtype=object)

In [26]:
import pandas as pd

columns = [
    "delivery_charges",
    "final_amount_inr","customer_city","customer_state","customer_tier",
    "customer_spending_tier","customer_age_group","payment_method","delivery_days",
    "delivery_type","is_prime_member","is_festival_sale",
    "festival_name"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

delivery_charges: ['0.0']

final_amount_inr: ['100002.6', '100004.92', '10001.79', '10002.34', '10003.8', '100031.2', '100034.82', '100038.1', '10005.26', '100050.6', '100052.34', '100054.81', '100057.75', '100061.2', '100062.63', '100066.53', '100066.70999999999', '100081.79999999999', '100084.47', '100086.04', '100088.89', '10009.07', '100090.5', '100106.0', '100110.55', '10012.69', '10013.85', '100133.97', '10014.27', '100144.25', '100158.14', '100158.48', '100158.92', '100159.68', '100163.86', '100168.4', '100173.41', '100174.96', '10018.25', '10018.47', '100183.05', '100186.23', '100192.43', '100195.76', '100196.99', '1002.02', '10021.05', '10021.17', '100212.75', '10022.24', '100225.13', '100226.35', '100226.4', '100233.51', '100235.04', '10025.09', '10025.57', '100251.58', '100251.69', '100255.67', '100258.15', '100267.32', '10027.79', '100277.81', '100278.42', '10028.06', '10028.71', '100292.7', '100293.02', '100296.01', '100308.41', '10031.28', 

In [27]:
import pandas as pd

columns = [    "customer_rating","return_status","order_month","order_year","order_quarter",
    "product_weight_kg","is_prime_eligible","product_rating"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

customer_rating: ['3.0', '3.5', '4.0', '4.5', '5.0']

return_status: ['Cancelled', 'Delivered', 'Returned']

order_month: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']

order_year: ['2019']

order_quarter: ['1', '2', '3', '4']

product_weight_kg: ['0.03', '0.04', '0.05', '0.06', '0.07', '0.08', '0.09', '0.1', '0.12', '0.13', '0.14', '0.15', '0.16', '0.17', '0.18', '0.19', '0.2', '0.21', '0.22', '0.23', '0.24', '0.25', '0.28', '0.29', '0.3', '0.31', '0.32', '0.33', '0.34', '0.35', '0.38', '0.4', '0.41', '0.42', '0.43', '0.45', '0.46', '0.47', '0.48', '0.49', '0.52', '0.53', '0.54', '0.55', '0.56', '0.57', '0.58', '0.59', '0.62', '0.63', '0.64', '0.65', '0.66', '0.67', '0.68', '0.69', '0.71', '0.72', '0.73', '0.75', '0.78', '1.2', '1.21', '1.24', '1.29', '1.37', '1.39', '1.4', '1.46', '1.5', '1.51', '1.57', '1.62', '1.63', '1.64', '1.65', '1.7', '1.72', '1.73', '1.74', '1.76', '1.78', '1.79', '1.81', '1.82', '1.85', '1.86', '1.88', '1.93'

In [28]:
dfc

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2019_00000001,2019-05-01,CUST_2019_00037860,PROD_001930,Fitbit Sports Watch Premium,Electronics,Smart Watch,Fitbit,29122.09,0.00,...,False,NaN,3.0,Delivered,1,2019,1,0.06,True,3.4
1,TXN_2019_00000002,2019-09-01,CUST_2019_00002481,PROD_001683,Apple Mi Pad 8GB RAM Silver,Electronics,Tablets,Apple,66396.45,0.00,...,False,NaN,3.5,Delivered,1,2019,1,0.40,True,4.5
2,TXN_2019_00000003,NaN,CUST_2018_00031347,PROD_000460,Oppo F9 64GB Black,Electronics,Smartphones,Oppo,29966.28,0.00,...,False,NaN,5.0,Delivered,1,2019,1,0.16,True,3.3
3,TXN_2019_00000004,NaN,CUST_2019_00024783,PROD_000633,Oppo Reno 64GB White,Electronics,Smartphones,Oppo,34225.21,19.68,...,True,Republic Day Sale,4.5,Delivered,1,2019,1,0.20,True,3.3
4,TXN_2019_00000005,NaN,CUST_2019_00018471,PROD_000571,Xiaomi Mi A3 256GB Blue,Electronics,Smartphones,Xiaomi,27092.49,0.00,...,False,NaN,NaN,Returned,1,2019,1,0.25,True,4.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121600,TXN_2019_00099999_DUP,NaN,CUST_2018_00006220,PROD_000347,Apple iPhone XS 64GB Blue,Electronics,Smartphones,Apple,149833.69,0.00,...,False,NaN,NaN,Delivered,11,2019,4,0.21,True,4.3
121601,TXN_2019_00116598_DUP,2019-12-12,CUST_2019_00042574,PROD_001745,Realme Tab M10 4GB RAM Silver,Electronics,Tablets,Realme,36655.57,0.00,...,False,NaN,5.0,Delivered,12,2019,4,0.57,True,3.1
121602,TXN_2019_00106497_DUP,2019-05-11,CUST_2015_00010945,PROD_000084,Xiaomi Redmi Note 4G 32GB Blue,Electronics,Smartphones,Xiaomi,33240.58,68.83,...,True,Diwali Sale,4.0,Delivered,11,2019,4,0.19,True,4.2
121603,TXN_2019_00027813_DUP,NaN,CUST_2016_00011885,PROD_000061,OnePlus OnePlus X 32GB White,Electronics,Smartphones,OnePlus,88975.63,23.47,...,False,NaN,NaN,Delivered,4,2019,2,0.19,True,3.6


In [29]:
print(len(dfc.columns))
print(dfc.columns.tolist())


34
['transaction_id', 'order_date', 'customer_id', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'original_price_inr', 'discount_percent', 'discounted_price_inr', 'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr', 'customer_city', 'customer_state', 'customer_tier', 'customer_spending_tier', 'customer_age_group', 'payment_method', 'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name', 'customer_rating', 'return_status', 'order_month', 'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible', 'product_rating']


In [30]:
dfc[dfc.duplicated()]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating


In [31]:
import pandas as pd

decimal_cols = dfc.select_dtypes(include=['float', 'float64']).columns

dfc[decimal_cols] = dfc[decimal_cols].round(2)
print(decimal_cols)


Index(['original_price_inr', 'discount_percent', 'discounted_price_inr',
       'subtotal_inr', 'delivery_charges', 'final_amount_inr',
       'customer_rating', 'product_weight_kg', 'product_rating'],
      dtype='object')


In [32]:
dfc.to_csv(r"C:\Users\hp\Desktop\Amazon\CSV_Clean_Files\amazon_india_2019_clean.csv",header='infer',index=False)